# Didattica con ChatGPT

## Setup

In [ ]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install openai

In [1]:
import openai
import os

client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])  # this is also the default, it can be omitted
client

In [2]:
# EV: old
# def get_completion(prompt, model="gpt-3.5-turbo", temperature=0): 
#     messages = [{"role": "user", "content": prompt}]
#     response = openai.ChatCompletion.create(
#         model=model,
#         messages=messages,
#         temperature=temperature, 
#     )
#     return response.choices[0].message["content"]

def ev_completion(client, model_par: str, prompt_par: str):
    """
    https://stackoverflow.com/questions/75774873/openai-chatgpt-gpt-3-5-api-error-this-is-a-chat-model-and-not-supported-in-t
    """
    try:
        completion_pyd = client.chat.completions.create( # Change the method name
            model = model_par,
            messages = [ # Change the prompt parameter to messages parameter
                {'role': 'user', 'content': prompt_par}
            ],
            temperature = 0  
        )
    except Exception as exc:
        print(exc)
        return None
    
    completion_d = completion_pyd.model_dump()
    reply0_txt = completion_d['choices'][0]['message']['content']
    return reply0_txt, completion_d, completion_pyd


def get_completion(prompt, model="gpt-3.5-turbo"):
    global client
    ret = ev_completion(client, model, prompt)
    return ret[0]

In [3]:
def generate_prompt(materia1: str, materia2 : str, lista_temi, 
    nr_domande = 15, nr_risposte = 5):

    temi = ", ".join(lista_temi)

    prompt =  f"""
    genera {nr_domande} domande di {materia1} nell'ambito di {materia2} sui temi seguenti: {temi}.

    Per ogni domanda genera {nr_risposte} risposte, una sola corretta. 
    Nelle risposte includi sempre l’opzione “nessuna è corretta”.
    Nelle risposte includi sempre l’opzione “altro, specificare”.
    Per ogni domanda genera anche una variante semplificata per studenti con i seguenti disturbi {lista_disturbi}.
    Il formato deve essere JSON, con le seguenti chiavi:
    - domanda
    - risposte_l, per la lista delle risposte
    - risposta corretta
    """
    return prompt

materia1 = "Italiano"
materia2 = "letteratura"
lista_temi = ["Manzoni", "Leopardi"]
nr_domande = 10
nr_risposte = 5
lista_disturbi = ["dislessia"]


prompt =  generate_prompt(materia1, materia2, lista_temi)

print(prompt)


    genera 15 domande di Italiano nell'ambito di letteratura sui temi seguenti: Manzoni, Leopardi.

    Per ogni domanda genera 5 risposte, una sola corretta. 
    Nelle risposte includi sempre l’opzione “nessuna è corretta”.
    Nelle risposte includi sempre l’opzione “altro, specificare”.
    Per ogni domanda genera anche una variante semplificata per studenti con i seguenti disturbi ['dislessia'].
    Il formato deve essere JSON, con le seguenti chiavi:
    - domanda
    - risposte_l, per la lista delle risposte
    - risposta corretta
    


In [4]:
response = get_completion(prompt)
print(response)

[
    {
        "domanda": "In quale città nacque Alessandro Manzoni?",
        "risposte_l": ["Milano", "Roma", "Firenze", "Napoli", "nessuna è corretta"],
        "risposta_corretta": "Milano"
    },
    {
        "domanda": "Quale opera di Alessandro Manzoni è considerata il capolavoro del Romanticismo italiano?",
        "risposte_l": ["I Promessi Sposi", "Il Gattopardo", "La coscienza di Zeno", "I Malavoglia", "nessuna è corretta"],
        "risposta_corretta": "I Promessi Sposi"
    },
    {
        "domanda": "Quale poeta italiano è noto per la sua profonda malinconia e pessimismo?",
        "risposte_l": ["Alessandro Manzoni", "Giacomo Leopardi", "Ugo Foscolo", "Giosuè Carducci", "nessuna è corretta"],
        "risposta_corretta": "Giacomo Leopardi"
    },
    {
        "domanda": "In quale città visse gran parte della sua vita Giacomo Leopardi?",
        "risposte_l": ["Roma", "Firenze", "Napoli", "Torino", "nessuna è corretta"],
        "risposta_corretta": "Napoli"
    },
  

In [5]:
import json

domande = json.loads(response)
domande

[{'domanda': 'In quale città nacque Alessandro Manzoni?',
  'risposte_l': ['Milano', 'Roma', 'Firenze', 'Napoli', 'nessuna è corretta'],
  'risposta_corretta': 'Milano'},
 {'domanda': 'Quale opera di Alessandro Manzoni è considerata il capolavoro del Romanticismo italiano?',
  'risposte_l': ['I Promessi Sposi',
   'Il Gattopardo',
   'La coscienza di Zeno',
   'I Malavoglia',
   'nessuna è corretta'],
  'risposta_corretta': 'I Promessi Sposi'},
 {'domanda': 'Quale poeta italiano è noto per la sua profonda malinconia e pessimismo?',
  'risposte_l': ['Alessandro Manzoni',
   'Giacomo Leopardi',
   'Ugo Foscolo',
   'Giosuè Carducci',
   'nessuna è corretta'],
  'risposta_corretta': 'Giacomo Leopardi'},
 {'domanda': 'In quale città visse gran parte della sua vita Giacomo Leopardi?',
  'risposte_l': ['Roma', 'Firenze', 'Napoli', 'Torino', 'nessuna è corretta'],
  'risposta_corretta': 'Napoli'},
 {'domanda': 'Qual è il titolo della raccolta di poesie più famosa di Giacomo Leopardi?',
  'ris

In [6]:
import random

def generate_domande(domande):

    ret = ""
    RISPOSTA_CORRETTA_KEY = "risposta corretta"

    for nr, domanda_d in enumerate(domande, start=1):
        domanda = domanda_d['domanda']
        # print(f"{nr:0>2} {domanda}")
        ret += f"{nr:0>2} {domanda}"+"\n"
        # random.shuffle(domanda_d['risposte_l']) # mette risp. corretta sempre stesso post
        for risposta in domanda_d['risposte_l']:
            if "corretta" in risposta.lower() and "risposta" in risposta.lower():
                continue # non stamparla
            ret += f"o {risposta}"+"\n"
        if RISPOSTA_CORRETTA_KEY in domanda_d:
            risposta_corretta = domanda_d[RISPOSTA_CORRETTA_KEY]
            ret += "### corretta: "+risposta_corretta+"\n"
            #.split(":")[1].strip()
            # print(risposta_corretta)
            # pass
        else:
            print(domanda_d.keys())

    return ret

In [7]:
ret = generate_domande(domande)
print(ret)


dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['domanda', 'risposte_l', 'risposta_corretta'])
dict_keys(['do

In [8]:
prompt = f"""
format the text delimited by ### in an HTML table
put both the questions and each answer on its own line.
Make the table higly readable for font colours
###
{ret}
###
"""

html_table = get_completion(prompt)
html_table

'<table>\n<tr>\n<td style="color: blue;">01 In quale città nacque Alessandro Manzoni?</td>\n<td>o Milano</td>\n</tr>\n<tr>\n<td style="color: blue;">o Roma</td>\n<td>o Firenze</td>\n</tr>\n<tr>\n<td></td>\n<td>o Napoli</td>\n</tr>\n<tr>\n<td></td>\n<td>o nessuna è corretta</td>\n</tr>\n<tr>\n<td style="color: blue;">02 Quale opera di Alessandro Manzoni è considerata il capolavoro del Romanticismo italiano?</td>\n<td>o I Promessi Sposi</td>\n</tr>\n<tr>\n<td></td>\n<td>o Il Gattopardo</td>\n</tr>\n<tr>\n<td></td>\n<td>o La coscienza di Zeno</td>\n</tr>\n<tr>\n<td></td>\n<td>o I Malavoglia</td>\n</tr>\n<tr>\n<td></td>\n<td>o nessuna è corretta</td>\n</tr>\n<tr>\n<td style="color: blue;">03 Quale poeta italiano è noto per la sua profonda malinconia e pessimismo?</td>\n<td>o Alessandro Manzoni</td>\n</tr>\n<tr>\n<td></td>\n<td>o Giacomo Leopardi</td>\n</tr>\n<tr>\n<td></td>\n<td>o Ugo Foscolo</td>\n</tr>\n<tr>\n<td></td>\n<td>o Giosuè Carducci</td>\n</tr>\n<tr>\n<td></td>\n<td>o nessuna è 

In [9]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(html_table))

01 In quale città nacque Alessandro Manzoni?,o Milano
o Roma,o Firenze
,o Napoli
,o nessuna è corretta
02 Quale opera di Alessandro Manzoni è considerata il capolavoro del Romanticismo italiano?,o I Promessi Sposi
,o Il Gattopardo
,o La coscienza di Zeno
,o I Malavoglia
,o nessuna è corretta
03 Quale poeta italiano è noto per la sua profonda malinconia e pessimismo?,o Alessandro Manzoni
,o Giacomo Leopardi


## Traduzione

ChatGPT è addestrato con fonti in molte lingue. Ciò dà al modello la capacità di eseguire traduzioni. Ecco alcuni esempi di come utilizzare questa funzionalità.

In [10]:
prompt = f"""
Traduci il testo Inglese seguente in Italiano: \ 
```Hi, I would like to order a beer```
"""
response = get_completion(prompt)
print(response)

Ciao, vorrei ordinare una birra.


In [15]:
prompt = f"""
dimmi in che linguaggio è il testo delimitato da ```: 
```Combien coûte le lampadaire?```
"""
response = get_completion(prompt)
print(response)

Il testo delimitato da ``` è in lingua francese.


In [11]:
prompt = f"""
traduci il testo delimitato da ``` in Francese, Spagnolo e English pirate: \
```I want to order a basketball```
"""
response = get_completion(prompt)
print(response)

- Francese: "Je veux commander un ballon de basket"
- Spagnolo: "Quiero pedir un balón de baloncesto"
- English pirate: "I be wantin' to order a basketball"


In [12]:
prompt = f"""
Traduci il testo seguente in Italiano, sia in linguggio fromale sia colloquiale: 
'Would you like to order a pillow?'
"""
response = get_completion(prompt)
print(response)

Formale: Desidererebbe ordinare un cuscino?
Colloquiale: Ti va di ordinare un cuscino?


### Traduttore universale  
Immagina di essere responsabile dell'IT presso una grande azienda multinazionale di e-commerce. Gli utenti ti inviano messaggi relativi a problemi IT in tutte le loro lingue native. Il tuo personale proviene da tutto il mondo e parla solo la propria lingua madre. Hai bisogno di un traduttore universale!

In [13]:
user_messages = [
  "La performance du système est plus lente que d'habitude.",  # System performance is slower than normal         
  "Mi monitor tiene píxeles que no se iluminan.",              # My monitor has pixels that are not lighting
  "Il mio mouse non funziona",                                 # My mouse is not working
  "Mój klawisz Ctrl jest zepsuty",                             # My keyboard has a broken control key
  "我的屏幕在闪烁"                                               # My screen is flashing
] 

In [14]:
for issue in user_messages:
    prompt = f"Tell me what language this is: ```{issue}```"
    lang = get_completion(prompt)
    print(f"\n\nOriginal message ({lang}): {issue}")

    prompt = f"""
    Translate the following  text to English \
    and Korean: ```{issue}```
    """
    response = get_completion(prompt)
    print(response)



Original message (French): La performance du système est plus lente que d'habitude.
English: "The system performance is slower than usual."

Korean: "시스템 성능이 평소보다 느립니다."


Original message (This is Spanish.): Mi monitor tiene píxeles que no se iluminan.
English: "My monitor has pixels that do not light up."

Korean: "내 모니터에는 빛나지 않는 픽셀이 있습니다."


Original message (Italian): Il mio mouse non funziona
English: My mouse is not working
Korean: 내 마우스가 작동하지 않습니다


Original message (This is Polish.): Mój klawisz Ctrl jest zepsuty
English: My Ctrl key is broken
Korean: 제 Ctrl 키가 고장 났어요


Original message (This is Chinese.): 我的屏幕在闪烁
English: My screen is flickering
Korean: 내 화면이 깜박거립니다


## Trasformazione del tono  
La scrittura può variare in base al pubblico a cui si rivolge. ChatGPT può produrre toni diversi.

In [15]:
prompt = f"""
Traduci il testo seguente dallo slang in una lettera commerciale: 

'Ehi fratello, dai un'occhiata alle specifiche di questa cavolo di lampada da tavolo,
che io mi sono proprio rotto, non ce la faccio.
E ricordate che aspetto il dinero, nun ce dormi sopra'
"""
response = get_completion(prompt)
# print(response)
print("\n".join(response.split(".")))

Gentile cliente,

Le invio le specifiche della lampada da tavolo che desidera
 La mia attuale lampada è proprio rotta e non ce la faccio più
 Le ricordo che attendo il pagamento, non si faccia trovare impreparato


Cordiali saluti



## Conversioni di formato  
ChatGPT può tradurre tra formati "tecnici". Il prompt dovrebbe descrivere i formati di input e output.

In [16]:
data_json = { "resturant employees" :[ 
    {"name":"Shyam", "email":"shyamjaiswal@gmail.com"},
    {"name":"Bob", "email":"bob32@gmail.com"},
    {"name":"Jai", "email":"jai87@gmail.com"}
]}

prompt = f"""
Translate the following python dictionary from JSON to an HTML \
table with column headers and title: {data_json}
"""
response = get_completion(prompt)
print(response)

<html>
<head>
    <title>Restaurant Employees</title>
</head>
<body>
    <table border="1">
        <tr>
            <th>Name</th>
            <th>Email</th>
        </tr>
        <tr>
            <td>Shyam</td>
            <td>shyamjaiswal@gmail.com</td>
        </tr>
        <tr>
            <td>Bob</td>
            <td>bob32@gmail.com</td>
        </tr>
        <tr>
            <td>Jai</td>
            <td>jai87@gmail.com</td>
        </tr>
    </table>
</body>
</html>


In [17]:
from IPython.display import display, Markdown, Latex, HTML, JSON
display(HTML(response))

Name,Email
Shyam,shyamjaiswal@gmail.com
Bob,bob32@gmail.com
Jai,jai87@gmail.com


## Controllo di ortografia e grammatica  

Ecco alcuni esempi di problemi grammaticali e ortografici comuni e la risposta del LLM.

Per segnalare al LLM che desideri che corregga il tuo testo, istruisci il modello a "correggere" o "correggere e correggere".

In [22]:
text = [ 
  "The girl with the black and white puppies have a ball.",  # The girl has a ball.
  "Yolanda has her notebook.", # ok
  "Its going to be a long day. Does the car need it’s oil changed?",  # Homonyms
  "Their goes my freedom. There going to bring they’re suitcases.",  # Homonyms
  "Your going to need you’re notebook.",  # Homonyms
  "That medicine effects my ability to sleep. Have you heard of the butterfly affect?", # Homonyms
  "This phrase is to cherck chatGPT for speling abilitty"  # spelling
]

text = [ 
  "Ieri ho andato alla mare",
  "Quel collega è troppo chiacchieroni.",
]



for t in text:
    prompt = f"""Controlla e correggi il testo seguente, e scrivi la versione corretta.
    Se non trovi errori, dì semplicemente "Nessun errore trovato".
    Non utilizzare segni di punteggiatura attorno al testo:
    ```{t}```"""
    response = get_completion(prompt)
    print(response)

Ieri sono andato al mare.
Nessun errore trovato.


In [23]:
responses_l = []

for t in text:
    prompt = f"""
    Rileggi e correggi il testo delimitato da ```.
    se il testo contiene errori
    - scrivere sia la versione originale che quella corretta, ciascuna sulla propria riga.
    - anteporre alla versione originale <originale>,
    - anteporre alla versione corretta <corretta>
    - scrivere una terza riga che spieghi l'errore, con il prefisso <spiegazione>
    Se il testo non contiene errori:
    - scrivere il testo su una riga
    - ha preceduto il testo con <nessun errore>
    
    Non utilizzare simboli di punteggiatura o virgolette attorno al testo.
    
    Il testo è questo:
    ```{t}```"""
    response = get_completion(prompt)
    responses_l.append(response)

In [24]:
for response in responses_l:
    print("-"*4+"\n"+response+"\n")

----
<originale>
Ieri ho andato alla mare
<corretta>
Ieri sono andato al mare
<spiegazione>
Errore di concordanza verbale: "andato" deve concordare in genere e numero con il soggetto "io". Inoltre, si usa "al mare" anziché "alla mare".

----
<originale>
Quel collega è troppo chiacchieroni.
<corretta>
Quel collega è troppo chiacchierone.
<spiegazione>
Il termine "chiacchieroni" è al plurale, mentre dovrebbe essere al singolare per concordare con "collega".



In [25]:
responses_l = []

for t in text:
    prompt = f"""
    Rileggi e correggi il testo delimitato da ```.
    L'output deve essere in formato JSON.
    Le chiavi dovrebbero essere le seguenti:
    - "corretto", booleano
    - "originale": il testo invariato
    - "corretto": presente solo quando il testo contiene errori
    - "spiegazione": presente solo quando il testo contiene errori

    Non utilizzare simboli di punteggiatura o virgolette attorno al testo.

    Il testo è questo:
    ```{t}```"""
    response = get_completion(prompt)
    responses_l.append(response)

In [26]:
for response in responses_l:
    print("-"*4+"\n"+response+"\n")

----
{
    "corretto": false,
    "originale": "Ieri ho andato alla mare",
    "corretto": "Ieri sono andato al mare",
    "spiegazione": "Corretta la forma del verbo 'andare' e la preposizione 'al' anziché 'alla'"
}

----
{
    "corretto": false,
    "originale": "Quel collega è troppo chiacchieroni.",
    "corretto": "Quel collega è troppo chiacchierone.",
    "spiegazione": "La parola 'chiacchieroni' è al plurale, ma dovrebbe essere al singolare 'chiacchierone' per concordare con 'collega'."
}



In [34]:
text = f"""
Got this for my daughter for her birthday cuz she keeps taking \
mine from my room.  Yes, adults also like pandas too.  She takes \
it everywhere with her, and it's super soft and cute.  One of the \
ears is a bit lower than the other, and I don't think that was \
designed to be asymmetrical. It's a bit small for what I paid for it \
though. I think there might be other options that are bigger for \
the same price.  It arrived a day earlier than expected, so I got \
to play with it myself before I gave it to my daughter.
"""

text = f"""
L'ho prendetti per mio figlia per il suo compleanno perché continua a prendere \
il mio dalla mia stanza. Sì, anche agli adulti ci piacciono i panda. Lei lo prende \
e lo portano ovunque con lei, ed è super morbido e caruccia. \
Le recchie sono un po' più basse delle altre, e non credo che sarebbe \
progettato per essere asimmetrico. È un po' piccolo per quello che l'ho pagato\
È arrivato un giorno prima del previsto, quindi ho ricevuto \
per giocarcine io stesso prima di darlo a mia figlia."""


prompt = f"proofread and correct this review: ```{text}```"
response = get_completion(prompt)
response = "\n".join(response.split("."))
print(response)

L'ho preso per mia figlia per il suo compleanno perché continuava a prendere il mio dalla mia stanza
 Sì, anche agli adulti piacciono i panda
 Lei lo prende e lo porta ovunque con lei, ed è super morbido e carino
 Le orecchie sono un po' più basse delle altre, e non credo che sia stato progettato per essere asimmetrico
 È un po' piccolo per quello che ho pagato
 È arrivato un giorno prima del previsto, quindi ho potuto giocarci io stesso prima di darlo a mia figlia



In [36]:
# Install a pip package in the current Jupyter kernel
import sys
!{sys.executable} -m pip install redlines

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/97.9 kB ? eta -:--:--
   ---------------------------------------- 97.9/97.9 kB 1.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/240.7 kB ? eta -:--:--
   ---------------------------------------- 240.7/240.7 kB 7.4 MB/s eta 0:00:00


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
python-lsp-black 1.2.1 requires black>=22.3.0, but you have black 0.0 which is incompatible.


In [38]:
from redlines import Redlines

diff = Redlines(text,response)
display(Markdown(diff.output_markdown))

L'ho <span style='color:red;font-weight:700;text-decoration:line-through;'>prendetti </span><span style='color:green;font-weight:700;'>preso </span>per <span style='color:red;font-weight:700;text-decoration:line-through;'>mio </span><span style='color:green;font-weight:700;'>mia </span>figlia per il suo compleanno perché <span style='color:red;font-weight:700;text-decoration:line-through;'>continua </span><span style='color:green;font-weight:700;'>continuava </span>a prendere il mio dalla mia <span style='color:red;font-weight:700;text-decoration:line-through;'>stanza. </span><span style='color:green;font-weight:700;'>stanza </span>

<span style='color:green;font-weight:700;'></span>Sì, anche agli adulti <span style='color:red;font-weight:700;text-decoration:line-through;'>ci </span>piacciono i <span style='color:red;font-weight:700;text-decoration:line-through;'>panda. </span><span style='color:green;font-weight:700;'>panda </span>

<span style='color:green;font-weight:700;'></span>Lei lo prende e lo <span style='color:red;font-weight:700;text-decoration:line-through;'>portano </span><span style='color:green;font-weight:700;'>porta </span>ovunque con lei, ed è super morbido e <span style='color:red;font-weight:700;text-decoration:line-through;'>caruccia. </span><span style='color:green;font-weight:700;'>carino </span>

<span style='color:green;font-weight:700;'></span>Le <span style='color:red;font-weight:700;text-decoration:line-through;'>recchie </span><span style='color:green;font-weight:700;'>orecchie </span>sono un po' più basse delle altre, e non credo che <span style='color:red;font-weight:700;text-decoration:line-through;'>sarebbe </span><span style='color:green;font-weight:700;'>sia stato </span>progettato per essere <span style='color:red;font-weight:700;text-decoration:line-through;'>asimmetrico. </span><span style='color:green;font-weight:700;'>asimmetrico </span>

<span style='color:green;font-weight:700;'></span>È un po' piccolo per quello che <span style='color:red;font-weight:700;text-decoration:line-through;'>l'ho pagatoÈ </span><span style='color:green;font-weight:700;'>ho pagato </span>

<span style='color:green;font-weight:700;'>È </span>arrivato un giorno prima del previsto, quindi ho <span style='color:red;font-weight:700;text-decoration:line-through;'>ricevuto per giocarcine </span><span style='color:green;font-weight:700;'>potuto giocarci </span>io stesso prima di darlo a mia <span style='color:red;font-weight:700;text-decoration:line-through;'>figlia.</span><span style='color:green;font-weight:700;'>figlia</span>

In [39]:
prompt = f"""
rileggi e correggi questa recensione. Rendilo più avvincente.
Assicurati che segua la guida di stile APA e sia rivolto a un lettore avanzato.
Output in formato markdown.
Testo: ```{text}```
"""
response = get_completion(prompt)
display(Markdown(response))

**Recensione del peluche Panda**

Ho acquistato questo adorabile peluche Panda per il compleanno di mia figlia, stufa di rubarmi il mio dalla mia stanza. Anche noi adulti non possiamo resistere al fascino di questi teneri animaletti! Il peluche è diventato subito il suo compagno inseparabile, portandolo ovunque con sé. La morbidezza e la dolcezza del peluche sono irresistibili, anche se devo ammettere che le orecchie sono leggermente asimmetriche, forse un piccolo difetto di fabbricazione. Nonostante ciò, il peluche è davvero carino.

L'unica pecca è che, considerando le dimensioni, il prezzo potrebbe essere un po' troppo elevato. Tuttavia, devo dire che sono rimasto piacevolmente sorpreso quando il peluche è arrivato un giorno prima del previsto. Questo mi ha permesso di giocarci un po' prima di consegnarlo alla mia piccola. In definitiva, un acquisto che ha reso felice mia figlia e che mi ha regalato un momento di divertimento inaspettato.

# Fine  